# ReAct and tools

This notebook covers the "ReAct" section of Chapter 7, "Custom Modules, Types, and Adapters" (under "Built-In Modules", page 166): building a tool-calling agent with `dspy.ReAct`. ReAct follows a Thought-Action-Observation loop, automatically generates tool descriptions from docstrings, and handles context window overflow.

**Environment variables**: `OPENAI_API_KEY` (loaded from `.env`).

**Models**: `openai/gpt-5.6-luna`, the book's default.

**Cost**: a few short model calls. The README estimates $0.50–2.00 for all Chapter 7 notebooks together.

**Install**: the next cell installs the repository's pinned packages. If you set up the environment with `uv sync --frozen` as described in the Preface, everything is already installed and you can skip it; running it anyway is harmless. With pip or Google Colab, run the notebook from its `chapter07/` folder inside the cloned repository so that `../requirements.txt` resolves.

In [ ]:
%pip install -r ../requirements.txt -q


In [ ]:
import os
import dspy
from dotenv import load_dotenv

load_dotenv()

dspy.configure(lm=dspy.LM("openai/gpt-5.6-luna"))


## A small support agent with two tools

Tools can be plain Python functions, callable objects, or `dspy.Tool` instances. DSPy auto-generates tool descriptions from docstrings, so write good ones.


In [ ]:
def lookup_order(order_id: str) -> str:
    """Look up order status by order ID."""
    return f"Order {order_id}: Shipped, arriving Thursday"

def check_return_policy(product: str) -> str:
    """Check return policy for a product category."""
    return f"{product}: 30-day return window, must be unopened"

support = dspy.ReAct(
    "customer_query: str -> response: str",
    tools=[lookup_order, check_return_policy]
)

> **Book vs. notebook:** The listing on page 166 ends once the agent is constructed. The cell below calls it with a question that needs both tools and prints the response.

In [ ]:
result = support(
    customer_query="Where is order #A1042, and can I return the headphones I ordered if they don't fit?"
)
print(result.response)